<a href="https://colab.research.google.com/github/Prudhvilakshman1112/GEN-AI/blob/main/EXP_4_Text_Generation_using_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Environment Setup and Initialization
In this cell, we install (if necessary) and import the core libraries. We use set_seed to ensure that our random text generations are reproducible. We also detect if a GPU (CUDA) is available to speed up the generation process.

In [ ]:
from transformers import pipeline, set_seed, GPT2LMHeadModel, GPT2Tokenizer
import torch

# Reproducibility for exact generations
set_seed(42)

# Check for GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

#Cell 2: Text Generation via the Pipeline API
The pipeline is the easiest way to use a pre-trained model. Here, we load the gpt2 model for the task of text-generation. This API handles tokenization, model inference, and decoding automatically. We set parameters like temperature (to control randomness) and top_p (to ensure the model chooses from the most likely words).

In [ ]:
# Initialize the high-level pipeline
generator = pipeline('text-generation', model='gpt2', device=device)

prompt = "Once upon a time in a distant galaxy,"

# Generate 3 variations of the story
outputs = generator(prompt, max_new_tokens=100, num_return_sequences=3,
                    temperature=0.8, top_p=0.95, do_sample=True, pad_token_id=50256)

print("=== Text Generations from Pipeline ===")
for i, output in enumerate(outputs):
    print(f"\nGeneration {i+1}:\n{output['generated_text']}")
    print("-" * 80)

#Cell 3: Loading the Tokenizer and Model Manually
For more complex tasks, you might want to interact with the Tokenizer and Model separately. The Tokenizer converts text into numerical IDs (tokens) that the model understands, while the Model processes those numbers. We explicitly set the pad_token to the end-of-sentence token to avoid errors during generation.

In [ ]:
# Manual approach: Loading specific components
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to('cuda' if device == 0 else 'cpu')

# Ensure the tokenizer has a padding token defined
tokenizer.pad_token = tokenizer.eos_token

print("Model and Tokenizer loaded manually.")

#Cell 4: Manual Inference and Decoding
In this final cell, we manually convert a text prompt into tensors (numerical arrays) and pass them to the model.generate() method. This gives us access to advanced parameters like repetition_penalty, which prevents the model from repeating the same phrases. Finally, we "decode" the numerical output back into human-readable text.

In [ ]:
prompt2 = "In a world where AI has become conscious,"

# Tokenize input text
inputs = tokenizer(prompt2, return_tensors="pt").to(model.device)

# Generate output with fine-tuned parameters
outputs2 = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2
)

print("\n=== Manual Generation with Full Control ===")
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))